# Custom AI Agent with Long-Term Memory

**Problem:** chat assistants forget across sessions. This notebook prototypes an agent that keeps a short-term scratchpad for the current turn and a long-term vector store for things worth remembering, plus a reflection step at the end of a session to distill durable facts.

**Data:** 10 synthetic session JSONs in `../../data/conversations/`. These were generated with a faker so the *content* is nonsense. What's real is the schema (`session_id`, `user_id`, `intent`, `turns[]`). For the actual demo conversation I run a small hand-written transcript so retrieval has something meaningful to land on.

**Stack:** ChromaDB (in-memory) for long-term, LangChain agent with `ConversationBufferWindowMemory` (sliding window of the last k turns) for short-term, Gemini's OpenAI-compatible endpoint via `langchain-openai`.

**Note:** no real Redis here. I originally had Redis in the plan for short-term, but a windowed in-process buffer covers the same role for this size of demo, so adding Redis would just be ceremony. I also tried `ConversationSummaryBufferMemory` first, but it calls `llm.get_num_tokens_from_messages` for token counting and the Gemini LangChain wrapper doesn't implement that, so it threw on every turn. The windowed buffer sidesteps token counting entirely.

In [1]:
# !pip install langchain langchain-openai langchain-community chromadb -q

In [2]:
import os
import re
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = Path("../../data/conversations")
assert DATA_DIR.exists(), "Expected synthetic conversation traces in data/conversations/"

def strip_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s

In [3]:
sessions = []
for f in sorted(DATA_DIR.glob("*.json")):
    with open(f) as fp:
        sessions.append(json.load(fp))

print(f"Loaded {len(sessions)} sessions")
if sessions:
    s0 = sessions[0]
    print("Schema:", list(s0.keys()))
    print(f"First session: {s0.get('session_id')} | intent={s0.get('intent')} | turns={len(s0.get('turns', []))}")
    # The faker content is gibberish — useful as schema check, not as retrieval signal.
    print("Sample turn:", s0["turns"][0]["content"][:80])

Loaded 10 sessions
Schema: ['session_id', 'user_id', 'intent', 'turns']
First session: sess_000 | intent=feature request | turns=8
Sample turn: Pretty region ability live car difficult.


## Preprocessing — embed turns and store in ChromaDB

In [4]:
import chromadb
from openai import OpenAI

chroma_client = chromadb.Client()  # in-memory
collection = chroma_client.get_or_create_collection("long_term_memory")

# Use the raw OpenAI client directly. LangChain's OpenAIEmbeddings tokenizes inputs
# with tiktoken and sends token-ID arrays, which Gemini's OpenAI-compat endpoint
# rejects (501 UNIMPLEMENTED). Plain text strings work fine.
embed_client = OpenAI()
EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "gemini-embedding-001")

def embed_text(text: str):
    resp = embed_client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

def store_memory(session_id, text, metadata=None):
    if not text or not text.strip():
        return
    emb = embed_text(text)
    doc_id = f"{session_id}_{collection.count()}"
    collection.add(
        ids=[doc_id],
        embeddings=[emb],
        documents=[text],
        metadatas=[{"session_id": session_id, **(metadata or {})}],
    )

def retrieve_memories(query, n=5):
    if collection.count() == 0:
        return []
    emb = embed_text(query)
    results = collection.query(query_embeddings=[emb], n_results=min(n, collection.count()))
    return results["documents"][0]

# Seed with a small hand-written set so retrieval has something meaningful to find.
# (The faker-generated sessions stay in the schema demo above but aren't useful as memory.)
SEED_MEMORIES = [
    ("user_demo", "User's name is Rushmitha and she prefers Python over Java."),
    ("user_demo", "User is working on ML portfolio projects, especially LLM agents."),
    ("user_demo", "User dislikes verbose framework boilerplate and prefers small dependencies."),
    ("user_demo", "User mentioned a previous project on fraud drift monitoring."),
    ("user_demo", "User asked about long-term memory architectures last session."),
]
for sid, text in SEED_MEMORIES:
    store_memory(sid, text, metadata={"source": "seed"})

print(f"Stored {collection.count()} memory chunks")
print("Retrieval check:", retrieve_memories("what language does the user like?", n=2))

Stored 5 memory chunks


Retrieval check: ["User's name is Rushmitha and she prefers Python over Java.", 'User dislikes verbose framework boilerplate and prefers small dependencies.']


## Modeling — LangChain agent with memory

In [5]:
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import initialize_agent, Tool, AgentType

llm = ChatOpenAI(model="gemini-flash-lite-latest", temperature=0.3)

# Sliding-window buffer over the last k turns. No token counting, which is what
# blew up with ConversationSummaryBufferMemory on Gemini (the wrapper doesn't
# implement get_num_tokens_from_messages and tiktoken refuses to count cl100k
# for a non-OpenAI model).
session_memory = ConversationBufferWindowMemory(
    k=6,
    return_messages=True,
    memory_key="chat_history",
)

def recall_tool(query: str) -> str:
    memories = retrieve_memories(query, n=3)
    if not memories:
        return "No relevant memories found."
    return "Recalled memories:\n" + "\n".join(f"- {m}" for m in memories)

def save_memory_tool(text: str) -> str:
    store_memory("agent_session", text, metadata={"source": "agent_save"})
    return f"Saved to long-term memory: {text[:60]}..."

tools = [
    Tool(name="RecallMemory", func=recall_tool,
         description="Recall relevant facts about the user from past sessions. Input: a query string."),
    Tool(name="SaveMemory",   func=save_memory_tool,
         description="Save an important new fact about the user to long-term memory. Input: the fact as a string."),
]

# initialize_agent + CONVERSATIONAL_REACT_DESCRIPTION is deprecated in newer langchain
# but still works and is the cleanest path for a memory-tool demo. LangGraph would be
# the modern rewrite, noted in the next-steps section.
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=session_memory,
    verbose=False,
    handle_parsing_errors=True,
)

print("Agent ready")

Agent ready


/var/folders/6_/qqrf82qn33v6prn9sqhbl1k40000gn/T/ipykernel_8841/339413550.py:37: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [6]:
# Small demo conversation. The seed memories above mean the agent should be
# able to surface real facts via the RecallMemory tool.
# Note: this used to throw `get_num_tokens_from_messages not implemented for cl100k_base`
# on every turn before I swapped the memory class. Rerun after the swap.
turns = [
    "Hi, can you recall what you know about my language preferences?",
    "What kind of projects have I worked on before?",
    "Please remember that I'm currently exploring LangGraph for agent workflows.",
]

for turn in turns:
    print(f"User: {turn}")
    try:
        resp = agent.run(turn)
    except Exception as e:
        # ReAct parsers occasionally choke on chatty Gemini output, surface and continue.
        resp = f"(agent parse error: {e})"
    print(f"Agent: {resp}\n")

/var/folders/6_/qqrf82qn33v6prn9sqhbl1k40000gn/T/ipykernel_8841/618697704.py:14: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  resp = agent.run(turn)


User: Hi, can you recall what you know about my language preferences?


Agent: Hi Rushmitha! Based on our previous interactions, I know that you prefer Python over Java. You also have a preference for small dependencies and dislike verbose framework boilerplate. Is there anything else you'd like to add or update regarding your language or development preferences?

User: What kind of projects have I worked on before?


Agent: Based on our previous conversations, Rushmitha, you have worked on a project involving fraud drift monitoring. You are also currently focused on building out your machine learning portfolio, with a particular interest in developing LLM agents. Does that sound right, or is there another project you'd like to add to your memory?

User: Please remember that I'm currently exploring LangGraph for agent workflows.


Agent: Got it, Rushmitha! I've saved that to your memory. Exploring LangGraph is a great step, especially given your interest in building LLM agents. How are you finding it so far—are you working on a specific workflow or just getting a feel for the state management?



## Reflection Loop — summarize session into durable memory

In [7]:
def reflect_and_store(session_id):
    """At session end, ask the LLM to distill key facts and store them."""
    history = session_memory.load_memory_variables({})
    history_text = str(history.get("chat_history", history.get("history", "")))

    if not history_text.strip():
        print("Nothing in session memory to reflect on.")
        return []

    prompt = (
        "From the following conversation, extract 3-5 key facts worth remembering "
        'long-term about the user. Return JSON: {"facts": ["...", "..."]}.\n\n'
        f"Conversation:\n{history_text[:2000]}"
    )
    resp = llm.invoke(prompt)
    raw = strip_fences(resp.content)
    try:
        obj = json.loads(raw)
        facts = obj.get("facts", []) if isinstance(obj, dict) else list(obj)
    except Exception:
        # Fall back to one-line-per-fact if the model ignored the JSON request.
        facts = [line.strip("-* ").strip() for line in raw.splitlines() if line.strip()]
        facts = facts[:5]

    for fact in facts:
        store_memory(session_id, fact, metadata={"type": "reflection"})

    print(f"Stored {len(facts)} reflection memories for session '{session_id}'")
    return facts

facts = reflect_and_store("session_001")
for f in facts:
    print(f"  - {f}")

Stored 5 reflection memories for session 'session_001'
  - The user's name is Rushmitha.
  - She prefers Python over Java and dislikes verbose framework boilerplate, favoring small dependencies.
  - She has experience with fraud drift monitoring projects.
  - She is currently building a machine learning portfolio with a focus on LLM agents.
  - She is currently exploring LangGraph for agent workflows.


## Memory Controls — inspect, edit, delete

In [8]:
all_mems = collection.get()
print(f"Total memories: {len(all_mems['ids'])}\n")
for mid, doc, meta in zip(all_mems["ids"][:8], all_mems["documents"][:8], all_mems["metadatas"][:8]):
    src = meta.get("source") or meta.get("type") or "-"
    print(f"  [{mid}] ({src}) {doc[:80]}")

# Delete one and verify count drops.
if all_mems["ids"]:
    victim = all_mems["ids"][0]
    before = collection.count()
    collection.delete(ids=[victim])
    after = collection.count()
    print(f"\nDeleted {victim}: {before} -> {after}")

# Quick recall sanity check: should still find a language-preference memory.
print("Recall after delete:", retrieve_memories("user's coding language preference", n=2))

Total memories: 11

  [user_demo_0] (seed) User's name is Rushmitha and she prefers Python over Java.
  [user_demo_1] (seed) User is working on ML portfolio projects, especially LLM agents.
  [user_demo_2] (seed) User dislikes verbose framework boilerplate and prefers small dependencies.
  [user_demo_3] (seed) User mentioned a previous project on fraud drift monitoring.
  [user_demo_4] (seed) User asked about long-term memory architectures last session.
  [agent_session_5] (agent_save) Rushmitha is currently exploring LangGraph for agent workflows.
  [session_001_6] (reflection) The user's name is Rushmitha.
  [session_001_7] (reflection) She prefers Python over Java and dislikes verbose framework boilerplate, favorin

Deleted user_demo_0: 11 -> 10
Recall after delete: ['She prefers Python over Java and dislikes verbose framework boilerplate, favoring small dependencies.', 'User dislikes verbose framework boilerplate and prefers small dependencies.']


## Notes from building this

A few things I'd flag if I were shipping this:

- The `conversations/` JSONs are faker output, useful only to validate schema-handling code. The retrieval demo runs on a tiny hand-seeded set instead, which is honest but obviously not a benchmark. Real next step is to either generate domain-realistic synthetic dialogues with an LLM, or hook this up to an actual chat product and log real sessions (with consent).
- `initialize_agent` is deprecated. LangGraph is the modern replacement and gives explicit control over what's persisted vs what's only in the scratchpad. Worth the rewrite for anything beyond a prototype.
- I'm using `ConversationBufferWindowMemory(k=6)` for short-term, which is the dumbest possible thing that works. Once a turn ages out of the window it's gone unless the reflection step caught it. The summary-buffer version would be nicer but it needs a token counter that works with Gemini, which is a side quest I didn't have time for.
- The reflection step is a single LLM call per session right now. Forgetting / decay (drop low-utility memories over time) and dedup (don't store five paraphrases of the same fact) are the obvious missing pieces.
- Privacy: `collection.delete(ids=[...])` is one line and works, but a real product needs user-scoped collections and a "forget everything about me" path that also hits any caches.